# Compare Complexity Levels Before and During Changes

This notebook compares complexity levels from before to during a change at equal window sizes.

The output is a table showing the effect of different change types (add, remove, move, and combinations) on various complexity measures.


In [ ]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
import ast
from typing import Dict, List, Tuple, Optional
import sys

# Add project root to path
sys.path.insert(0, str(Path().resolve().parent.parent.parent))

from utils import helpers, constants
from scripts.change_study.computations.combine_results import get_synthetic_datasets_by_noise_level


## Configuration

Define the window size and noise levels to analyze.


In [ ]:
# Configuration
WINDOW_SIZE = 200  # e.g., 200 or 400
OVERLAP = 20  # Overlap size (typically 20)

# Noise levels to analyze (automatically processes all three if True)
ANALYZE_NO_NOISE = True
ANALYZE_20_NOISE = True
ANALYZE_40_NOISE = True

# Window setting name format: groundtruth__fixed_w{window_size}_o{overlap}
WINDOW_SETTING = f"groundtruth__fixed_w{WINDOW_SIZE}_o{OVERLAP}"

print(f"Window setting: {WINDOW_SETTING}")


## Helper Functions


In [ ]:
def parse_list_string(s: str) -> List[str]:
    """Parse a string representation of a list into an actual list."""
    if pd.isna(s) or s == '[]' or s == '':
        return []
    
    # Convert to string and strip whitespace
    s = str(s).strip()
    
    # Remove outer quotes if present (e.g., '"[item]"' -> '[item]')
    if s.startswith('"') and s.endswith('"'):
        s = s[1:-1]
    if s.startswith("'") and s.endswith("'"):
        s = s[1:-1]
    
    if s == '[]' or s == '':
        return []
    
    try:
        # Handle both '[item]' and '["item1", "item2"]' formats
        parsed = ast.literal_eval(s)
        if isinstance(parsed, list):
            return [str(item) for item in parsed]  # Ensure all items are strings
        return [str(parsed)] if parsed else []
    except (ValueError, SyntaxError):
        return []


def determine_change_type(row: pd.Series) -> str:
    """Determine the change type from a ground truth row.
    
    Returns: 'add', 'remove', 'move', or 'multiple' (combination of types)
    """
    activities_added = parse_list_string(str(row.get('activities_added', '')))
    activities_deleted = parse_list_string(str(row.get('activities_deleted', '')))
    activities_moved = parse_list_string(str(row.get('activities_moved', '')))
    
    has_add = len(activities_added) > 0
    has_remove = len(activities_deleted) > 0
    has_move = len(activities_moved) > 0
    
    change_types = []
    if has_add:
        change_types.append('add')
    if has_remove:
        change_types.append('remove')
    if has_move:
        change_types.append('move')
    
    if len(change_types) > 1:
        return 'multiple'
    elif len(change_types) == 1:
        return change_types[0]
    else:
        return 'unknown'


def get_complexity_at_index(complexity_df: pd.DataFrame, target_index: int) -> Optional[pd.Series]:
    """Get complexity values for the window that contains the target index.
    
    Returns None if no window contains the target index.
    """
    # Find window where first_index <= target_index <= last_index
    mask = (complexity_df['first_index'] <= target_index) & (complexity_df['last_index'] >= target_index)
    matching = complexity_df[mask]
    
    if matching.empty:
        return None
    
    # If multiple windows match, return the one with center_moment closest to target
    # For simplicity, return the first match (should be unique in practice)
    return matching.iloc[0]


def get_complexity_closest_to_index(complexity_df: pd.DataFrame, target_index: int) -> Optional[pd.Series]:
    """Get complexity values for the window with center index closest to target_index."""
    if complexity_df.empty:
        return None
    
    # Calculate center index for each window
    complexity_df = complexity_df.copy()
    complexity_df['center_index'] = (complexity_df['first_index'] + complexity_df['last_index']) / 2
    
    # Find closest
    complexity_df['distance'] = abs(complexity_df['center_index'] - target_index)
    closest_idx = complexity_df['distance'].idxmin()
    
    return complexity_df.loc[closest_idx]


## Main Processing Function


In [ ]:
def process_dataset(
    dataset_key: str,
    window_size: int,
    window_setting: str
) -> List[Dict]:
    """Process a single dataset and extract complexity changes.
    
    Returns a list of dictionaries, each containing:
    - dataset_key
    - change_type (add/remove/move/multiple)
    - measure_name
    - complexity_before
    - complexity_during
    - relative_change
    """
    results = []
    
    # Load ground truth drift detection
    gt_path = Path(f"results/change_study/drift_detection/{dataset_key}/results_{dataset_key}_groundtruth.csv")
    if not gt_path.exists():
        print(f"Warning: Ground truth file not found for {dataset_key}")
        return results
    
    gt_df = pd.read_csv(gt_path)
    
    # Load complexity assessment
    complexity_path = Path(f"results/change_study/complexity_assessment/{dataset_key}/{window_setting}/complexity.csv")
    if not complexity_path.exists():
        print(f"Warning: Complexity file not found for {dataset_key} with setting {window_setting}")
        return results
    
    complexity_df = pd.read_csv(complexity_path)
    
    # Get measure columns (all columns starting with 'measure_')
    measure_columns = [col for col in complexity_df.columns if col.startswith('measure_')]
    
    if not measure_columns:
        print(f"Warning: No measure columns found in complexity file for {dataset_key}")
        return results
    
    # Group changes by calc_process_change_id to handle gradual changes
    # For gradual changes, we need gradual_start and gradual_end
    change_groups = {}
    for _, row in gt_df.iterrows():
        change_id = row.get('calc_process_change_id')
        change_type_str = str(row.get('calc_change_type', ''))
        
        if pd.isna(change_id):
            continue
        
        if change_id not in change_groups:
            change_groups[change_id] = {}
        
        if 'gradual_start' in change_type_str:
            change_groups[change_id]['start'] = row
        elif 'gradual_end' in change_type_str:
            change_groups[change_id]['end'] = row
        elif 'sudden' in change_type_str:
            change_groups[change_id]['sudden'] = row
    
    # Process each change
    for change_id, change_info in change_groups.items():
        # Determine if this is a gradual or sudden change
        if 'sudden' in change_info:
            # Sudden change
            change_row = change_info['sudden']
            change_index = int(change_row['calc_change_index'])
            
            # Get complexity before (change_index - window_size/2)
            before_index = max(0, int(change_index - window_size / 2))
            complexity_before = get_complexity_at_index(complexity_df, before_index)
            
            # Get complexity during (closest to change_index)
            complexity_during = get_complexity_closest_to_index(complexity_df, change_index)
            
        elif 'start' in change_info and 'end' in change_info:
            # Gradual change
            start_row = change_info['start']
            end_row = change_info['end']
            start_index = int(start_row['calc_change_index'])
            end_index = int(end_row['calc_change_index'])
            middle_index = int((start_index + end_index) / 2)
            
            # Get complexity before (start_index - window_size/2)
            before_index = max(0, int(start_index - window_size / 2))
            complexity_before = get_complexity_at_index(complexity_df, before_index)
            
            # Get complexity during (middle of gradual change)
            complexity_during = get_complexity_at_index(complexity_df, middle_index)
            
            # Use start_row for change type determination (both should have same info)
            change_row = start_row
        else:
            # Incomplete change information
            continue
        
        if complexity_before is None or complexity_during is None:
            continue
        
        # Determine change type
        change_type = determine_change_type(change_row)
        
        # Check if the before window overlaps with any other change
        # (This is a simplified check - in practice, you might want more sophisticated overlap detection)
        before_start = complexity_before['first_index']
        before_end = complexity_before['last_index']
        
        # Check for overlaps with other changes
        has_overlap = False
        for other_change_id, other_change_info in change_groups.items():
            if other_change_id == change_id:
                continue
            
            if 'sudden' in other_change_info:
                other_index = int(other_change_info['sudden']['calc_change_index'])
                if before_start <= other_index <= before_end:
                    has_overlap = True
                    break
            elif 'start' in other_change_info:
                other_start = int(other_change_info['start']['calc_change_index'])
                if before_start <= other_start <= before_end:
                    has_overlap = True
                    break
        
        if has_overlap:
            continue  # Skip this change if before window overlaps with another change
        
        # Calculate relative changes for each measure
        for measure_col in measure_columns:
            measure_name = measure_col.replace('measure_', '')
            
            before_val = complexity_before[measure_col]
            during_val = complexity_during[measure_col]
            
            # Skip if either value is NaN
            if pd.isna(before_val) or pd.isna(during_val):
                continue
            
            # Calculate relative change: (during - before) / before
            if before_val != 0:
                relative_change = (during_val - before_val) / before_val
            else:
                # Handle division by zero
                if during_val == 0:
                    relative_change = 0.0
                else:
                    relative_change = np.inf if during_val > 0 else -np.inf
            
            results.append({
                'dataset_key': dataset_key,
                'change_type': change_type,
                'measure_name': measure_name,
                'complexity_before': before_val,
                'complexity_during': during_val,
                'relative_change': relative_change
            })
    
    return results


## Process All Datasets


In [ ]:
# Collect all results
all_results = []

# Process each noise level
noise_levels_to_process = []
if ANALYZE_NO_NOISE:
    noise_levels_to_process.append("0")
if ANALYZE_20_NOISE:
    noise_levels_to_process.append("20")
if ANALYZE_40_NOISE:
    noise_levels_to_process.append("40")

for noise_level in noise_levels_to_process:
    print(f"\nProcessing noise level: {noise_level}%")
    datasets = get_synthetic_datasets_by_noise_level(noise_level)
    print(f"Found {len(datasets)} datasets")
    
    for dataset_key in datasets:
        print(f"  Processing {dataset_key}...", end=" ")
        results = process_dataset(dataset_key, WINDOW_SIZE, WINDOW_SETTING)
        all_results.extend(results)
        print(f"Found {len(results)} change-measure combinations")

print(f"\nTotal results collected: {len(all_results)}")


## Create Summary Table


In [ ]:
# Convert results to DataFrame
if all_results:
    results_df = pd.DataFrame(all_results)
    
    # Create pivot table: change_type (rows) x measure_name (columns) with average relative_change
    summary_table = results_df.pivot_table(
        values='relative_change',
        index='change_type',
        columns='measure_name',
        aggfunc='mean'
    )
    
    # Round to reasonable precision
    summary_table = summary_table.round(4)
    
    # Display the table
    print("Summary Table: Average Relative Change in Complexity by Change Type")
    print("=" * 80)
    display(summary_table)
    
    # Also show counts for reference
    counts_table = results_df.pivot_table(
        values='relative_change',
        index='change_type',
        columns='measure_name',
        aggfunc='count'
    )
    print("\nCounts (number of observations per cell):")
    display(counts_table)
else:
    print("No results to display. Check that:")
    print("1. Ground truth files exist in results/change_study/drift_detection/")
    print("2. Complexity files exist in results/change_study/complexity_assessment/")
    print("3. Window setting matches: " + WINDOW_SETTING)


## Export Results (Optional)


In [ ]:
# Uncomment to save results
# output_dir = Path("results/change_study/combined_results/complexity_change_comparison")
# output_dir.mkdir(parents=True, exist_ok=True)
# 
# if all_results:
#     # Save detailed results
#     results_df.to_csv(output_dir / f"detailed_results_w{WINDOW_SIZE}_o{OVERLAP}.csv", index=False)
#     
#     # Save summary table
#     summary_table.to_csv(output_dir / f"summary_table_w{WINDOW_SIZE}_o{OVERLAP}.csv")
#     
#     print(f"Results saved to {output_dir}")
